In [ ]:
# 02_fix_gold.ipynb
# Цель:
# 1) прочитать goldenset.xlsx
# 2) прочитать chunks_final.csv
# 3) для каждого вопроса найти все чанки той же компании и той же страницы
# 4) выгрузить gold_markup.xlsx
# 5) после ручной разметки прочитать gold_markup_filled.xlsx
# 6) проверить разметку
# 7) собрать gold_final.csv

In [ ]:
import re
from pathlib import Path

# Нужные библиотеки поставьте через pip, если их нет
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill

In [ ]:
GOLD_INPUT_PATH = "goldenset.xlsx"
CHUNKS_PATH = "chunks_final.csv"

MARKUP_OUTPUT_PATH = "gold_markup.xlsx"
MARKUP_FILLED_PATH = "gold_markup_filled.xlsx"
GOLD_FINAL_PATH = "gold_final.csv"

In [ ]:
def make_company_slug(name: str) -> str:
    """
    Нормализует название компании в slug.
    Должна совпадать с логикой из 01_build_chunks.ipynb - просто берём оттуда
    """
    # TODO: реализовать
    raise NotImplementedError

In [ ]:
gold_df = pd.read_excel(GOLD_INPUT_PATH)
chunks_df = pd.read_csv(CHUNKS_PATH)

print(gold_df.shape)
print(chunks_df.shape)

# можно .head() посмотреть ещё

In [ ]:
## Готовим таблицу вопросов для разметки 

required_gold_cols = ["company", "question", "answer", "pdf_page"]
missing_gold_cols = [c for c in required_gold_cols if c not in gold_df.columns]
assert not missing_gold_cols, f"В goldenset.xlsx не хватает колонок: {missing_gold_cols}"

gold_prepared = gold_df.copy()

gold_prepared["company"] = gold_prepared["company"].astype(str).str.strip()
gold_prepared["question"] = gold_prepared["question"].astype(str).str.strip()
gold_prepared["answer"] = gold_prepared["answer"].astype(str).str.strip()
gold_prepared["pdf_page"] = gold_prepared["pdf_page"].astype(int)

gold_prepared["query_id"] = [
    f"q{i:03d}" for i in range(1, len(gold_prepared) + 1)
]

gold_prepared["company_slug"] = gold_prepared["company"].apply(make_company_slug)

gold_prepared = gold_prepared[
    ["query_id", "company", "company_slug", "question", "answer", "pdf_page"]
]

gold_prepared.head(10)

In [ ]:
# Готовим таблицу чанков для разметки

required_chunk_cols = ["chunk_id", "company_slug", "pdf_page", "text"]
missing_chunk_cols = [c for c in required_chunk_cols if c not in chunks_df.columns]
assert not missing_chunk_cols, f"В chunks_final.csv не хватает колонок: {missing_chunk_cols}"

chunks_prepared = chunks_df.copy()

chunks_prepared["chunk_id"] = chunks_prepared["chunk_id"].astype(str).str.strip()
chunks_prepared["company_slug"] = chunks_prepared["company_slug"].astype(str).str.strip()
chunks_prepared["pdf_page"] = chunks_prepared["pdf_page"].astype(int)
chunks_prepared["text"] = chunks_prepared["text"].fillna("").astype(str).str.strip()

chunks_prepared = chunks_prepared[chunks_prepared["text"] != ""].copy()

chunks_prepared = chunks_prepared[
    ["chunk_id", "company_slug", "pdf_page", "text"]
].reset_index(drop=True)

chunks_prepared.head(10)

In [ ]:
# Собираем кандидатов для разметки

rows = []

for _, gold_row in gold_prepared.iterrows():
    query_id = gold_row["query_id"]
    company = gold_row["company"]
    company_slug = gold_row["company_slug"]
    question = gold_row["question"]
    answer = gold_row["answer"]
    pdf_page = gold_row["pdf_page"]

    page_chunks = chunks_prepared[
        (chunks_prepared["company_slug"] == company_slug) &
        (chunks_prepared["pdf_page"] == pdf_page)
    ].copy()

    page_chunks = page_chunks.sort_values("chunk_id").reset_index(drop=True)

    if len(page_chunks) == 0:
        rows.append({
            "query_id": query_id,
            "company": company,
            "company_slug": company_slug,
            "question": question,
            "answer": answer,
            "pdf_page": pdf_page,
            "candidate_order": None,
            "candidate_chunk_id": None,
            "candidate_text": None,
            "is_relevant": None,
            "review_comment": "NO_CHUNKS_FOUND"
        })
    else:
        for i, (_, chunk_row) in enumerate(page_chunks.iterrows(), start=1):
            rows.append({
                "query_id": query_id,
                "company": company,
                "company_slug": company_slug,
                "question": question,
                "answer": answer,
                "pdf_page": pdf_page,
                "candidate_order": i,
                "candidate_chunk_id": chunk_row["chunk_id"],
                "candidate_text": chunk_row["text"],
                "is_relevant": None,
                "review_comment": None
            })

markup_df = pd.DataFrame(rows)
markup_df.head(20)

In [ ]:
# Проверка
required_markup_cols = [
    "query_id",
    "company",
    "company_slug",
    "question",
    "answer",
    "pdf_page",
    "candidate_order",
    "candidate_chunk_id",
    "candidate_text",
    "is_relevant",
    "review_comment",
]

missing_markup_cols = [c for c in required_markup_cols if c not in markup_df.columns]
assert not missing_markup_cols, f"Не хватает колонок: {missing_markup_cols}"

print("Число строк в markup_df:", len(markup_df))
print("Число уникальных query_id:", markup_df["query_id"].nunique())

In [ ]:
# Экспортируем в Excel для ручной разметки

def export_markup_workbook(markup_df: pd.DataFrame, output_path: str) -> None:
    # Лист 00_readme
    readme_df = pd.DataFrame({
        "instruction": [
            "Откройте лист 01_candidates.",
            "Для каждого query_id поставьте 1 в колонке is_relevant ровно у одного чанка.",
            "Остальные строки оставьте пустыми.",
            "Если подходящего чанка нет, ничего не ставьте и напишите комментарий в review_comment.",
            "Не меняйте служебные колонки."
        ]
    })

    # Лист 02_validation — предварительная сводка
    validation_df = (
        markup_df.groupby("query_id", as_index=False)
        .agg(
            n_candidates=("candidate_chunk_id", lambda x: x.notna().sum())
        )
    )
    validation_df["n_selected"] = ""
    validation_df["status"] = ""

    # Сохраняем через pandas
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        readme_df.to_excel(writer, sheet_name="00_readme", index=False)
        markup_df.to_excel(writer, sheet_name="01_candidates", index=False)
        validation_df.to_excel(writer, sheet_name="02_validation", index=False)

    # Оформление через openpyxl
    wb = load_workbook(output_path)

    # ---------- 00_readme ----------
    ws_readme = wb["00_readme"]
    ws_readme.freeze_panes = "A2"
    ws_readme.column_dimensions["A"].width = 100
    for cell in ws_readme[1]:
        cell.font = Font(bold=True)
        cell.fill = PatternFill("solid", fgColor="D9EAF7")
    for row in ws_readme.iter_rows(min_row=2):
        for cell in row:
            cell.alignment = Alignment(wrap_text=True, vertical="top")

    # ---------- 01_candidates ----------
    ws_candidates = wb["01_candidates"]
    ws_candidates.freeze_panes = "A2"
    ws_candidates.auto_filter.ref = ws_candidates.dimensions

    widths = {
        "A": 10,   # query_id
        "B": 20,   # company
        "C": 20,   # company_slug
        "D": 40,   # question
        "E": 25,   # answer
        "F": 10,   # pdf_page
        "G": 15,   # candidate_order
        "H": 25,   # candidate_chunk_id
        "I": 80,   # candidate_text
        "J": 12,   # is_relevant
        "K": 30,   # review_comment
    }
    for col_letter, width in widths.items():
        ws_candidates.column_dimensions[col_letter].width = width

    for cell in ws_candidates[1]:
        cell.font = Font(bold=True)
        cell.fill = PatternFill("solid", fgColor="D9EAF7")

    for row in ws_candidates.iter_rows(min_row=2):
        for cell in row:
            cell.alignment = Alignment(wrap_text=True, vertical="top")

    # ---------- 02_validation ----------
    ws_validation = wb["02_validation"]
    ws_validation.freeze_panes = "A2"
    ws_validation.auto_filter.ref = ws_validation.dimensions

    for col_letter, width in {"A": 10, "B": 15, "C": 15, "D": 20}.items():
        ws_validation.column_dimensions[col_letter].width = width

    for cell in ws_validation[1]:
        cell.font = Font(bold=True)
        cell.fill = PatternFill("solid", fgColor="D9EAF7")

    wb.save(output_path)

In [ ]:
export_markup_workbook(markup_df, MARKUP_OUTPUT_PATH)
print(f"Сохранён файл: {MARKUP_OUTPUT_PATH}")

In [ ]:
# Далее открываем этот файл вручную и делаем разметку, а потом сохраняем как gold_markup_filled.xlsx

In [ ]:
# Теперь читаем разметку обратно и проверяем

filled_df = pd.read_excel(MARKUP_FILLED_PATH, sheet_name="01_candidates")

print("filled_df:", filled_df.shape)
filled_df.head(10)

In [ ]:
# Для каждого query_id нужно проверить:

# сколько всего кандидатов;
# сколько строк отмечено как is_relevant = 1;
# какой статус у вопроса.

# Можно упростить, но базово нужно, чтобы валидация проверяла эти моменты и возвращала summary-таблицу с колонками:

def validate_filled_markup(filled_df: pd.DataFrame) -> pd.DataFrame:
    """
    Проверяет заполненный файл разметки и возвращает summary-таблицу:
    - query_id
    - n_candidates
    - n_selected
    """
    # TODO: реализовать
    raise NotImplementedError

validation_summary = validate_filled_markup(filled_df)

validation_summary.head(20)
print(validation_summary["status"].value_counts(dropna=False))

In [ ]:
# Из filled_df нужно оставить только те вопросы, где выбран ровно один чанк, и собрать итоговую таблицу.

# На выходе должен получиться DataFrame с колонками:

# query_id
# company
# company_slug
# question
# answer
# pdf_page
# relevant_chunk_id

def build_gold_final(filled_df: pd.DataFrame) -> pd.DataFrame:
    """
    Собирает финальный gold set:
    одна строка = один вопрос = один relevant_chunk_id
    """
    # TODO: реализовать
    raise NotImplementedError



In [ ]:
gold_final_df = build_gold_final(filled_df)

print("gold_final_df:", gold_final_df.shape)
gold_final_df.head(10)

In [ ]:
# Проверяем gold_final_df
required_final_cols = [
    "query_id",
    "company",
    "company_slug",
    "question",
    "answer",
    "pdf_page",
    "relevant_chunk_id",
]

missing_final_cols = [c for c in required_final_cols if c not in gold_final_df.columns]
assert not missing_final_cols, f"В gold_final_df не хватает колонок: {missing_final_cols}"

assert gold_final_df["query_id"].is_unique, "query_id должны быть уникальны"
assert gold_final_df["relevant_chunk_id"].notna().all(), "Есть пустые relevant_chunk_id"
assert (gold_final_df["relevant_chunk_id"].astype(str).str.strip() != "").all(), \
    "Есть пустые relevant_chunk_id"

print("Проверка gold_final_df пройдена")

In [ ]:
# Сохраняем и тоже больше не трогаем

gold_final_df.to_csv(GOLD_FINAL_PATH, index=False, encoding="utf-8-sig")
print(f"Сохранён файл: {GOLD_FINAL_PATH}")